In [5]:
import cv2
import tkinter as tk
import numpy as np
import matplotlib

matplotlib.use("TkAgg")  # Ensure TkAgg backend is used for interactivity
import matplotlib.pyplot as plt
from matplotlib.widgets import RectangleSelector
from tkinter import filedialog, messagebox
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import find_peaks

# Global variables
crop_coords = None
pixel_size_um = None  # To store pixel size in micrometers


def line_select_callback(eclick, erelease):
    """Callback for RectangleSelector to store crop coordinates."""
    global crop_coords
    x1, y1 = int(eclick.xdata), int(eclick.ydata)
    x2, y2 = int(erelease.xdata), int(erelease.ydata)
    crop_coords = (min(x1, x2), max(x1, x2), min(y1, y2), max(y1, y2))


def load_image():
    file_path = filedialog.askopenfilename(
        title="Select an .npy file", filetypes=[("NumPy files", "*.npy")]
    )
    if file_path:
        image = np.load(file_path)
        return image, file_path
    else:
        messagebox.showinfo("Info", "No file selected.")
        return None, None


def calculate_pixel_size(image: np.ndarray, image_size_um: float) -> float:
    """Calculate the pixel size in micrometers based on the image dimensions."""
    global pixel_size_um
    # Calculate the pixel size
    image_width_pixels = image.shape[1]
    pixel_size_um = image_size_um / image_width_pixels
    print(f"Calculated pixel size: {pixel_size_um:.6f} µm")
    return pixel_size_um


def crop_image(image):
    global crop_coords
    crop_coords = None  # Reset crop coordinates

    fig, ax = plt.subplots()
    ax.imshow(image, cmap="gray")
    ax.set_title("Drag to select the cropping area, then close the window")

    rectangle_selector = RectangleSelector(
        ax,
        line_select_callback,
        interactive=True,
        useblit=True,
        button=[1],  # Only left mouse button
        minspanx=5,
        minspany=5,
        spancoords="pixels",
    )

    plt.show()  # Wait for the user to close the window

    if crop_coords is None:
        messagebox.showinfo("Info", "No cropping area selected.")
        return None

    x_min, x_max, y_min, y_max = crop_coords
    cropped_image = image[y_min:y_max, x_min:x_max]
    return cropped_image


def process_image(data: np.ndarray, height: int, width: int) -> None:
    global pixel_size_um

    # Convert the image to picoamperes for better readability
    image_pA = data * 1e12  # Convert from A to pA

    # Calculate the histogram with fine binning
    hist, bin_edges = np.histogram(image_pA.flatten(), bins=10000)

    # Use `find_peaks` to locate all local maxima in the histogram
    peaks, _ = find_peaks(
        hist, height=100
    )  # Adjust `height` to filter insignificant peaks
    peak_intensities = bin_edges[peaks]  # Intensity values corresponding to the peaks

    # Sort the peaks by intensity and filter out the peak at the very beginning (close to 0)
    sorted_peaks = sorted(zip(hist[peaks], peak_intensities), reverse=True)

    # Select the primary peak closest to 0
    primary_peak_value = sorted_peaks[0][1]

    # Find the index of the primary peak
    primary_peak_index = np.digitize(
        primary_peak_value, bin_edges
    )  # Find the bin index of the primary peak

    # Get the next 20 frequencies (counts) in the histogram after the primary peak
    next_frequencies = hist[
        primary_peak_index + 1 : primary_peak_index + 21
    ]  # Get the next 20 bins

    # Get the corresponding intensity values (bin edges) for those frequencies
    next_intensities = bin_edges[primary_peak_index + 1 : primary_peak_index + 21]

    # Remove values below the primary peak value (set them to 0 or NaN)
    image_pA_filtered = np.copy(image_pA)  # Create a copy of the image
    image_pA_filtered[image_pA_filtered < primary_peak_value] = (
        0  # Set values below primary peak to 0
    )

    # # Print the identified noise levels and next frequencies
    # print(f"Primary Noise Level: {primary_peak_value:.2f} pA")
    # print("Next 20 Frequency Values After Primary Peak:")
    # for i, (intensity, frequency) in enumerate(zip(next_intensities, next_frequencies)):
    #     print(
    #         f"Next Frequency {i + 1}: Intensity = {intensity:.2f} pA, Frequency = {frequency} occurrences"
    #     )

    # Intensity value of the 20th pixel
    intensity_20th_pixel = next_intensities[
        -1
    ]  # Last value in next_intensities (20th pixel)
   
    # print(f"Intensity Value of 20th Pixel: {intensity_20th_pixel:.2f} pA")

    # # Plot the original image and its histogram with marked frequencies
    # fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    # # Original Image
    # axes[0].imshow(
    #     image_pA_filtered,
    #     cmap="gray",
    #     extent=[0, image_pA_filtered.shape[1], 0, image_pA_filtered.shape[0]],
    # )
    # axes[0].set_title("Filtered Image (Values Below Primary Peak Removed)")
    # axes[0].set_xlabel("X Pixels")
    # axes[0].set_ylabel("Y Pixels")
    # axes[0].axis("on")

    # # Histogram
    # axes[1].hist(image_pA_filtered.flatten(), bins=500, color="blue", alpha=0.7)
    # axes[1].set_title("Histogram of Filtered Pixel Intensities (in pA)")
    # axes[1].set_xlabel("Pixel Intensity (pA)")
    # axes[1].set_ylabel("Frequency")

    # # Mark the primary peak
    # axes[1].axvline(
    #     primary_peak_value,
    #     color="red",
    #     linestyle="--",
    #     label=f"Primary Peak: {primary_peak_value:.2f} pA",
    # )

    # # Mark the next frequencies
    # for i, intensity in enumerate(next_intensities):
    #     axes[1].axvline(
    #         intensity,
    #         color="green",
    #         linestyle="--",
    #         label=f"Next Frequency {i + 1}: {intensity:.2f} pA",
    #     )

    # axes[1].legend()
    # plt.tight_layout()
    # plt.show()

    # Calculate the 95th percentile value for clipping
    percentile_96 = np.percentile(data, 96)

    # Clip the image at the 95th percentile
    clipped_image = np.clip(data, None, percentile_96)

    # Calculate the average current
    average_current = np.mean(data)

    # Convert the data to nanoamperes (divide by 10^-9)
    data_nA = average_current / 1e-9

    # Normalize data
    normalized_data = cv2.normalize(
        clipped_image, None, 0, 255, cv2.NORM_MINMAX
    ).astype(np.uint8)

    # Define the coverage threshold
    threshold = intensity_20th_pixel * 1e-12

    # Create a binary mask for coverage (1 = covered, 0 = uncovered)
    coverage_mask = (data > threshold).astype(int)

    # Calculate coverage percentage
    coverage_percentage = (np.sum(coverage_mask) / coverage_mask.size) * 100
    print(f"Coverage Percentage: {coverage_percentage:.2f}%")

    # Create a color map: Light blue for coverage, light grey for uncovered
    colors = ["black", "mistyrose"]
    cmap = LinearSegmentedColormap.from_list("coverage_map", colors, N=2)

    # Plot the coverage map
    x = np.linspace(0, 2, data.shape[1])  # X-axis (microns)
    y = np.linspace(0, 2, data.shape[0])  # Y-axis (microns)
    X, Y = np.meshgrid(x, y)

    # Apply Gamma Correction
    gamma = 0.7
    gamma_corrected_img = np.power(normalized_data / 255.0, gamma) * 255.0
    gamma_corrected_img = gamma_corrected_img.astype(np.uint8)

    # Apply Bilateral Filter
    bilateral_filtered = cv2.bilateralFilter(gamma_corrected_img, 9, 30, 75)

    # Apply Gaussian Blur
    blurred = cv2.GaussianBlur(bilateral_filtered, (3, 3), 3)

    # Apply Non-Local Means Denoising
    denoised = cv2.fastNlMeansDenoising(blurred, None, 10, 2, 21)

    # Adaptive Thresholding
    thresholded = cv2.adaptiveThreshold(
        denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 9, 1
    )

    # Contour Detection
    contours, _ = cv2.findContours(thresholded, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
    output_image = cv2.cvtColor(denoised, cv2.COLOR_GRAY2BGR)

    curves_length = 0
    circular_shapes_area = 0
    extended_shapes_area = 0
    circular_shapes = []
    curved_lines = []
    extended_shapes = []

    for contour in contours:
        if cv2.contourArea(contour) > 10:
            perimeter = cv2.arcLength(contour, closed=True)
            area = cv2.contourArea(contour)
            circularity = 4 * np.pi * area / (perimeter**2) if perimeter != 0 else 0

            # Create a mask for the contour
            mask = np.zeros_like(thresholded, dtype=np.uint8)
            cv2.drawContours(mask, [contour], -1, 255, -1)

            # Calculate the total number of pixels inside the contour
            total_pixels = cv2.countNonZero(mask)

            # Calculate the number of white pixels inside the contour
            white_pixels = cv2.countNonZero(cv2.bitwise_and(mask, thresholded))

            x, y, w, h = cv2.boundingRect(contour)
            aspect_ratio = float(w) / h

            if circularity > 0.2 and area <= 1000:
                # Check for white pixel coverage only for circular shapes
                if total_pixels > 0 and (white_pixels / total_pixels) >= 0.8:
                    continue  # Skip this contour if 80% or more of the pixels are white

                # Process circular shapes
                circular_shapes.append(contour)
                circular_shapes_area += area
                cv2.drawContours(output_image, [contour], -1, (0, 255, 255), 1)
            elif aspect_ratio > 1.8 and area <= 1000:
                # Check for white pixel coverage only for extended shapes
                if total_pixels > 0 and (white_pixels / total_pixels) >= 0.8:
                    continue  # Skip this contour if 80% or more of the pixels are white

                # Process extended shapes
                extended_shapes.append(contour)
                extended_shapes_area += area
                cv2.drawContours(output_image, [contour], -1, (0, 255, 255), 1)
            else:
                # Process curved lines without the white pixel check
                curved_lines.append(contour)
                curve_length = cv2.arcLength(contour, closed=True)
                curves_length += curve_length
                cv2.drawContours(output_image, [contour], -1, (0, 255, 0), 1)

    # Convert areas and lengths to micrometers
    circular_shapes_area_um2 = circular_shapes_area * (pixel_size_um**2)
    extended_shapes_area_um2 = extended_shapes_area * (pixel_size_um**2)
    curves_length_um = curves_length * pixel_size_um

    # # Visualize results
    # plt.figure(figsize=(15, 20))
    # plt.subplot(3, 3, 1)
    # plt.imshow(normalized_data, cmap="copper")
    # plt.axis("off")
    # plt.title("Normalized Data")

    # plt.subplot(3, 3, 2)
    # plt.imshow(gamma_corrected_img, cmap="copper")
    # plt.axis("off")
    # plt.title("Processed Data (Gamma Correction)")

    # plt.subplot(3, 3, 3)
    # plt.imshow(bilateral_filtered, cmap="copper")
    # plt.axis("off")
    # plt.title("Bilateral Filter")

    # plt.subplot(3, 3, 4)
    # plt.imshow(blurred, cmap="copper")
    # plt.axis("off")
    # plt.title("Gaussian Blur")

    # plt.subplot(3, 3, 5)
    # plt.imshow(denoised, cmap="copper")
    # plt.axis("off")
    # plt.title("Denoised Data")

    # plt.subplot(3, 3, 6)
    # plt.imshow(thresholded, cmap="copper")
    # plt.axis("off")
    # plt.title("Thresholded Data")

    # plt.subplot(3, 3, 7)
    # plt.imshow(output_image[..., ::-1])
    # plt.axis("off")
    # plt.title("Detected Shapes")

    # plt.tight_layout()
    # plt.show()

    # plt.figure(figsize=(6, 6))  # New figure for the contour plot
    # plt.contourf(
    #     X, Y, coverage_mask, levels=[-0.5, 0.5, 1.5], colors=colors, origin="lower"
    # )
    # plt.gca().invert_yaxis()  # Invert the Y-axis
    # plt.gca().xaxis.set_ticks_position("top")  # Move the X-axis to the top
    # plt.gca().xaxis.set_label_position("top")
    # plt.title("Coverage Map of MoS2 Surface")
    # plt.xlabel("X (microns)")
    # plt.ylabel("Y (microns)")
    # plt.colorbar(label="Coverage (0: Uncovered, 1: Covered)", ticks=[0, 1])
    # plt.show()

    curves_length_um_without_boundary = curves_length_um - (2 * (height + width))
    Total_Defect_Area = circular_shapes_area_um2 + extended_shapes_area_um2
    Total_Defect_Percentage = 100 * Total_Defect_Area / (height * width)
    
    print(f"Total Length of Detected Curves: {curves_length_um:.2f} \u03bcm")
    print(
        f"Total Length of Detected Curves without boundary: {curves_length_um_without_boundary:.2f} \u03bcm"
    )
    print(
        f"Total Area of Circular Shapes: {circular_shapes_area_um2:.2f} \u03bcm\u00b2"
    )
    print(
        f"Total Area of Extended Shapes: {extended_shapes_area_um2:.2f} \u03bcm\u00b2"
    )
    print(f"Total Defect Area: {Total_Defect_Area:.2f} \u03bcm\u00b2")
    print(f"Total Defect Percentage: {Total_Defect_Percentage:.2f} %")
    print(f"Number of Circular Shapes: {len(circular_shapes)}")
    print(f"Number of Extended Shapes: {len(extended_shapes)}")
    print(f"Number of Curved Lines: {len(curved_lines)}")
    print(f"Average Current of the Surface: {data_nA:.2f} nA")


def main():
    root = tk.Tk()
    root.withdraw()  # Hide the main window

    image, file_path = load_image()
    if image is None:
        return

    calculate_pixel_size(image)  # Calculate pixel size before cropping

    response = messagebox.askyesno("Crop Image", "Do you want to crop the image?")
    if response:
        cropped_image = crop_image(image)
        if cropped_image is not None:
            selected_image = cropped_image
            messagebox.showinfo("Info", "Cropped image selected.")
        else:
            return
    else:
        selected_image = image
        messagebox.showinfo("Info", "Whole image selected.")

    # Calculate area
    height, width = selected_image.shape
    area_um2 = height * width * (pixel_size_um**2)
    height_um = height * pixel_size_um
    width_um = width * pixel_size_um
    print(f"Area of Selected Image: {area_um2:.2f} μm²")
    print(f"Height of Selected Image: {height_um:.2f} μm")
    print(f"Width of Selected Image: {width_um:.2f} μm")

    process_image(selected_image, height_um, width_um)

In [6]:
fp = "/playpen/mufan/levi/tianlong-chen-lab/material-super-resolution/data/raw-data/1-23-25/img1cf1-MoS2-Sef-New-Area1-position 1 (again) (2um-good) (hivac)_250102_Current_Forward_014.npy"
data = np.load(fp)
height, width = 512, 512
calculate_pixel_size(data, 2.0)
process_image(data, height, width)

Calculated pixel size: 0.003906 µm
Coverage Percentage: 97.01%
Total Length of Detected Curves: 33.35 μm
Total Length of Detected Curves without boundary: -2014.65 μm
Total Area of Circular Shapes: 0.96 μm²
Total Area of Extended Shapes: 0.01 μm²
Total Defect Area: 0.97 μm²
Total Defect Percentage: 0.00 %
Number of Circular Shapes: 1230
Number of Extended Shapes: 3
Number of Curved Lines: 6
Average Current of the Surface: 9.41 nA
